In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
from sklearn.datasets import load_breast_cancer, make_classification
import numpy as np

data = load_breast_cancer()
X, y = data.data, data.target

# Small imbalanced dataset
X_small, y_small = make_classification(
    n_samples=25,
    n_classes=2,
    weights=[0.7, 0.3],
    random_state=42
)

print(f"Full Dataset: {X.shape}, Class Balance: {np.bincount(y)}")
print(f"Small Dataset: {X_small.shape}, Class Balance: {np.bincount(y_small)}")

Full Dataset: (569, 30), Class Balance: [212 357]
Small Dataset: (25, 20), Class Balance: [18  7]


## 1. Classical scikit-learn cross-validation

### 1.1 Stratified 5-Fold CV on the full dataset

**Description (conceptual)**
 This setup evaluates a model under a standard *stratified K-fold* regime:

- The breast cancer dataset `X, y` is split into 5 folds.
- In each fold, class proportions are preserved (stratification), which is important for slightly imbalanced binary problems.
- A RandomForest is trained on 4 folds and evaluated on the remaining fold.
- The process is repeated for all folds and the mean ± standard deviation of the accuracy is reported.

This gives a robust estimate of how well the model generalizes under an IID assumption.

In [2]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")

print(f"sklearn StratifiedKFold(5) accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

sklearn StratifiedKFold(5) accuracy: 0.956 ± 0.012


### 1.3 Leave-P-Out CV (LPOCV, P=2) on the small dataset

**Description (conceptual)**
 Leave-P-Out CV is similar to LOOCV but leaves *P* points out in each fold:

- Here `p=2`, so every possible pair of points is used as test set.
- For `n=25`, this yields `C(25,2)=300` folds.
- This is useful to understand stability under very small test sets but becomes combinatorially expensive for larger n.

In [3]:
from sklearn.model_selection import LeavePOut
from sklearn.linear_model import LogisticRegression

lpo = LeavePOut(p=2)
log_reg = LogisticRegression(max_iter=1000, solver="liblinear")

scores_lpo = cross_val_score(log_reg, X_small, y_small, cv=lpo, scoring="accuracy")

print(f"sklearn LPOCV (p=2) accuracy (X_small): {scores_lpo.mean():.3f}")
print(f"Number of folds (LPOCV): {len(scores_lpo)}")  # 300

sklearn LPOCV (p=2) accuracy (X_small): 0.913
Number of folds (LPOCV): 300


## 2. trustcv with TrustCVValidator (stratified K-fold + checks + CIs)

Here we switch from “manual” CV loops to the **TrustCVValidator** abstraction you provided. This concentrates:

- Choice of CV method (here: stratified K-fold).
- Leakage and balance checks.
- Metric aggregation (mean, std).
- Confidence interval estimation (t-interval or bootstrap).

### 2.1 StratifiedKFoldMedical via TrustCVValidator

**Description (conceptual)**

In this setup:

- We still use stratified 5-fold CV on the full breast cancer dataset `X, y`.
- The `TrustCVValidator` orchestrates:
  - Stratified splitting (`method="stratifiedKfold"`).
  - Optional leakage and class-balance checks per fold.
  - Computation of performance metrics (here at least accuracy by default; you can uncomment AUC, F1, etc.).
  - Computation of **confidence intervals** for each metric using the chosen CI method (`t-interval` at 95%).

The final `val_result.summary()` should print something like:

- `accuracy: mean ± std, 95% CI [lower, upper]`
- plus similar lines for any additional metrics you configure.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from trustcv import TrustCVValidator

model = RandomForestClassifier(random_state=42)

validator = TrustCVValidator(
    method="stratifiedKfold",   # IID, stratified CV
    n_splits=5,
    random_state=42,
    shuffle=True,
    check_leakage=True,
    check_balance=True,
    # metrics=["accuracy", "roc_auc", "f1"],  # optionally specify metrics
    return_confidence_intervals=True,
    ci_method="t-interval",     # 'bootstrap' or 't-interval'
    ci_level=0.95,              # confidence level
)

val_result = validator.validate(model=model, X=X, y=y)
print(val_result.summary())

=== Trustworthy Cross-Validation Results ===

Performance Metrics (mean +/- std) (method: t-interval):
  accuracy: 0.956 +/- 0.014 [95% CI (t-interval): 0.939-0.973]
  roc_auc: 0.989 +/- 0.009 [95% CI (t-interval): 0.977-1.000]
  sensitivity: 0.966 +/- 0.030 [95% CI (t-interval): 0.929-1.004]
  specificity: 0.939 +/- 0.056 [95% CI (t-interval): 0.869-1.009]
  precision: 0.965 +/- 0.031 [95% CI (t-interval): 0.926-1.004]
  recall: 0.966 +/- 0.030 [95% CI (t-interval): 0.929-1.004]
  f1: 0.965 +/- 0.011 [95% CI (t-interval): 0.952-0.979]

Data Integrity Checks:
  Leakage Check: PASSED
  Class Balance: PASSED



## 3. XGBoost CV on the full dataset

**Description (conceptual)**

This demonstrates how a boosting library’s **internal CV routine** works on the same data:

- `xgboost.cv` performs K-fold CV inside XGBoost itself (no manual loops).
- We use 5 stratified folds (`nfold=5`, `stratified=True`).
- The function returns a table of metrics (training and validation loss/metrics per boosting round).
- We typically look at the last row (or the best round) to report CV performance.

In [8]:
import xgboost as xgb

dtrain = xgb.DMatrix(X, label=y)

params = {
    "objective": "binary:logistic",
    "eval_metric": "error", # Changed from logloss to error
    "max_depth": 4,
    "eta": 0.1,
    "seed": 42,
}

cv_results_xgb_acc = xgb.cv(
    params=params,
    dtrain=dtrain,
    num_boost_round=200,
    nfold=5,
    stratified=True,
    seed=42,
    verbose_eval=False
)

# The 'error' metric is 1 - accuracy
mean_test_error = cv_results_xgb_acc['test-error-mean'].iloc[-1]
std_test_error = cv_results_xgb_acc['test-error-std'].iloc[-1]

mean_accuracy = 1 - mean_test_error
std_accuracy = std_test_error # Standard deviation of error is the same as standard deviation of accuracy

print(f"XGBoost 5-fold accuracy: {mean_accuracy:.3f} ± {std_accuracy:.3f}")
print("Full CV results (last row):")
print(cv_results_xgb_acc.tail(1))

XGBoost 5-fold accuracy: 0.960 ± 0.012
Full CV results (last row):
     train-error-mean  train-error-std  test-error-mean  test-error-std
199               0.0              0.0         0.040413        0.011872


## 4. LightGBM CV with sklearn-generated folds

**Description (conceptual)**

Here we decouple the *fold generation* from the learning algorithm:

- `StratifiedKFold` from sklearn defines which samples go in each train/test fold.
- These indices are passed to `lightgbm.cv` via the `folds` argument.
- This ensures:
  - Same CV strategy as other models (e.g. RandomForest, XGBoost, trustcv).
  - Direct comparability of results across different algorithms on identical folds.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

train_data = lgb.Dataset(X, label=y)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))

params = {
    "objective": "binary",
    "metric": "binary_error", # Changed metric to binary_error for accuracy
    "learning_rate": 0.05,
    "num_leaves": 31,
}

cv_results_lgb = lgb.cv(
    params=params,
    train_set=train_data,
    folds=folds,
    num_boost_round=200,
    seed=42
)

# The 'binary_error' metric is 1 - accuracy
mean_test_error_lgb = min(cv_results_lgb["valid binary_error-mean"])
std_test_error_lgb = cv_results_lgb["valid binary_error-stdv"][cv_results_lgb["valid binary_error-mean"].index(mean_test_error_lgb)]

mean_accuracy_lgb = 1 - mean_test_error_lgb
std_accuracy_lgb = std_test_error_lgb # Standard deviation of error is the same as standard deviation of accuracy

print(f"LightGBM 5-fold accuracy: {mean_accuracy_lgb:.3f} ± {std_accuracy_lgb:.3f}")


## 5. CatBoost CV on the full dataset

**Description (conceptual)**

CatBoost has its own CV mechanism:

- `catboost.cv` performs K-fold CV directly on a `Pool`.
- We use 5 folds, shuffling and stratifying labels.
- The output is a table of metrics (e.g. AUC) over iterations.
- As with XGBoost/LightGBM, we typically read off the final or best iteration.

In [ ]:
from catboost import Pool, cv

pool = Pool(X, y)

params = {
    "loss_function": "Logloss",
    "eval_metric": "Accuracy", # Changed from AUC to Accuracy
    "iterations": 200,
    "learning_rate": 0.05,
    "depth": 4,
    "random_seed": 42,
}

cv_results_cb = cv(
    pool=pool,
    params=params,
    fold_count=5,
    shuffle=True,
    stratified=True
)

# Extract and print accuracy results
mean_test_accuracy_cb = cv_results_cb['test-Accuracy-mean'].iloc[-1]
std_test_accuracy_cb = cv_results_cb['test-Accuracy-std'].iloc[-1]

print(f"CatBoost 5-fold accuracy: {mean_test_accuracy_cb:.3f} \u00b1 {std_test_accuracy_cb:.3f}")
print("Full CV results (last row):")
print(cv_results_cb.tail(1))

## 6. PyCaret with the breast-cancer dataset

**Description (conceptual)**

PyCaret abstracts away the explicit CV loop:

- We convert `X, y` into a `pandas.DataFrame`.
- `setup(..., fold=5, fold_strategy="stratifiedkfold")` configures stratified 5-fold CV globally.
- `compare_models` trains and evaluates many models using this CV configuration and returns top performers.
- This is a high-level interface: same dataset and CV strategy, but very little boilerplate.

In [ ]:
import pandas as pd
import pycaret.classification as pc

df = pd.DataFrame(X, columns=data.feature_names)
df["target"] = y

clf = pc.setup(
    data=df,
    target="target",
    fold=5,
    fold_strategy="stratifiedkfold",
    session_id=42,
    silent=True,
    verbose=False
)

best_models = pc.compare_models(sort="AUC", n_select=3)
print(best_models)

------

## 7. H2O with 5-fold stratified CV

**Description (conceptual)**

This shows how an H2O model performs internal cross-validation:

- The breast cancer data is wrapped in an `H2OFrame`.
- `nfolds=5` and `fold_assignment="Stratified"` instruct the estimator to perform stratified 5-fold CV inside H2O.
- `model.model_performance(xval=True)` returns cross-validated performance (here AUC).

In [5]:
import h2o
from h2o.estimators import H2ORandomForestEstimator
import pandas as pd
import numpy as np

h2o.init()

df_h2o = pd.DataFrame(X, columns=data.feature_names)
df_h2o["target"] = y
hf = h2o.H2OFrame(df_h2o)

hf["target"] = hf["target"].asfactor()

model = H2ORandomForestEstimator(
    nfolds=5,
    fold_assignment="Stratified",
    keep_cross_validation_predictions=True,
    seed=42
)

model.train(x=list(data.feature_names), y="target", training_frame=hf)

# Get overall cross-validation accuracy
cv_metrics_summary = model.cross_validation_metrics_summary().as_data_frame()

# Assuming the first column (index 0) contains the metric names like 'accuracy', 'auc', 'err'
# and subsequent columns are 'mean', 'sd', etc.
accuracy_row = cv_metrics_summary[cv_metrics_summary.iloc[:, 0] == 'accuracy']
mean_accuracy_h2o = accuracy_row['mean'].values[0]
std_accuracy_h2o = accuracy_row['sd'].values[0]

print(f"H2O CV Accuracy: {mean_accuracy_h2o:.3f} \u00b1 {std_accuracy_h2o:.3f}")
print("H2O CV AUC (overall):", model.model_performance(xval=True).auc())

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,4 mins 42 secs
H2O_cluster_timezone:,Europe/Stockholm
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.8
H2O_cluster_version_age:,1 month and 16 days
H2O_cluster_name:,H2O_from_python_amir_0biw15
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.824 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
drf Model Build progress: |██████████████████████████████████████████████████████| (done) 100%
H2O CV Accuracy: 0.975 ± 0.024
H2O CV AUC (overall): 0.9876856402938534


## 8. Keras (TensorFlow) with manual stratified K-fold

**Description (conceptual)**

This demonstrates how deep learning fits into the same CV structure:

- We define a small MLP in Keras.
- We reuse sklearn’s `StratifiedKFold` to generate 5 folds on `X, y`.
- For each fold:
  - A fresh model is built and trained.
  - Accuracy on the held-out fold is recorded.
- The result is again `mean ± std` accuracy across folds.

In [25]:
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import os
import random
import numpy as np
import tensorflow as tf

# Optional: reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

def make_keras_model(input_dim):
    model = Sequential([
        Dense(16, activation="relu", input_shape=(input_dim,)),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=Adam(0.001),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs = []

for train_idx, test_idx in skf.split(X, y):
    model = make_keras_model(X.shape[1])
    model.fit(X[train_idx], y[train_idx],
              epochs=20, batch_size=32, verbose=0)
    loss, acc = model.evaluate(X[test_idx], y[test_idx], verbose=0)
    accs.append(acc)

accs = np.array(accs)
print(f"Keras 5-fold accuracy: {accs.mean():.3f} ± {accs.std():.3f}")

Keras 5-fold accuracy: 0.863 ± 0.057


### Keras (TensorFlow) with stratified K-fold and  `TrustCVValidator`



In [ ]:
from scikeras.wrappers import KerasClassifier

# Optional: reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)


def make_keras_model(input_dim):
    model = Sequential([
        Dense(16, activation="relu", input_shape=(input_dim,)),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(0.001), loss="binary_crossentropy", metrics=["accuracy"])
    return model


model = KerasClassifier(
    model=make_keras_model,
    model__input_dim=X.shape[1],
    epochs=20,
    batch_size=32,
    verbose=0,
)

validator = TrustCVValidator(
    method="StratifiedKFold",   # IID, stratified CV
    n_splits=5,
    random_state=42,
    shuffle=True,
    check_leakage=True,
    check_balance=True,
    # metrics=["accuracy", "roc_auc", "f1"],  # optionally specify metrics
    return_confidence_intervals=True,
    ci_method="t-interval",     # 'bootstrap' or 't-interval'
    ci_level=0.95,              # confidence level
)
val_result = validator.validate(model=model, X=X, y=y)
print(val_result.summary())


=== Trustworthy Cross-Validation Results ===

Performance Metrics (mean +/- std) (method: t-interval):
  accuracy: 0.863 +/- 0.064 [95% CI (t-interval): 0.783-0.943]
  roc_auc: 0.883 +/- 0.037 [95% CI (t-interval): 0.837-0.929]
  sensitivity: 0.899 +/- 0.099 [95% CI (t-interval): 0.776-1.022]
  specificity: 0.802 +/- 0.061 [95% CI (t-interval): 0.726-0.879]
  precision: 0.885 +/- 0.033 [95% CI (t-interval): 0.844-0.925]
  recall: 0.899 +/- 0.099 [95% CI (t-interval): 0.776-1.022]
  f1: 0.890 +/- 0.057 [95% CI (t-interval): 0.819-0.961]

Data Integrity Checks:
  Leakage Check: PASSED
  Class Balance: PASSED



## 9. PyTorch with KFold on the same data

**Description (conceptual)**

Finally, we show the low-level deep learning case with PyTorch:

- `KFold` from sklearn defines 5 folds on `X, y`.
- The data is wrapped in a `TensorDataset`.
- For each fold:
  - We create new `DataLoader`s using index-based samplers for train and test.
  - A simple feed-forward network is trained for a fixed number of epochs.
  - Accuracy on the held-out fold is computed.
- Again we get `accuracy_mean ± accuracy_std` across the 5 folds.

In [6]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, SubsetRandomSampler
from sklearn.model_selection import KFold
import random

# Optional: reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed) 

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)
dataset = TensorDataset(X_t, y_t)

class SimpleNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()

def evaluate_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = (model(xb) > 0.5).float()
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accs_torch = []

for train_idx, test_idx in kf.split(X_t):
    model = SimpleNet(X_t.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.BCELoss()

    train_loader = DataLoader(dataset, batch_size=32,
                              sampler=SubsetRandomSampler(train_idx))
    test_loader = DataLoader(dataset, batch_size=32,
                             sampler=SubsetRandomSampler(test_idx))

    for _ in range(20):
        train_one_epoch(model, train_loader, optimizer, loss_fn)

    acc = evaluate_accuracy(model, test_loader)
    accs_torch.append(acc)

accs_torch = np.array(accs_torch)
print(f"PyTorch 5-fold accuracy: {accs_torch.mean():.3f} ± {accs_torch.std():.3f}")

PyTorch 5-fold accuracy: 0.907 ± 0.040


#### using `TrustCV`

In [8]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, SubsetRandomSampler
from sklearn.base import BaseEstimator, ClassifierMixin
from trustcv import TrustCVValidator

# keep the global seeds at the top of the notebook
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

class TorchSKClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim=None, lr=1e-3, epochs=20, batch_size=32, device="cpu", seed=42):
        self.input_dim = input_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = device
        self.seed = seed
        self._gen = torch.Generator().manual_seed(seed)

    def _build_model(self, in_dim):
        return nn.Sequential(
            nn.Linear(in_dim, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        ).to(self.device)

    def fit(self, X, y):
        # do NOT reseed here; let the global RNG advance like your manual loop
        X_t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        y_t = torch.as_tensor(y, dtype=torch.float32, device=self.device).view(-1, 1)

        self.model_ = self._build_model(self.input_dim or X_t.shape[1])
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)
        loss_fn = nn.BCELoss()

        idx = torch.arange(X_t.shape[0])
        loader = DataLoader(
            TensorDataset(X_t, y_t),
            batch_size=self.batch_size,
            sampler=SubsetRandomSampler(idx, generator=self._gen),  # deterministic across runs
        )

        self.model_.train()
        for _ in range(self.epochs):
            for xb, yb in loader:
                opt.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                opt.step()
        return self

    def predict_proba(self, X):
        self.model_.eval()
        X_t = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            probs = self.model_(X_t).cpu().numpy().ravel()
        probs = np.clip(probs, 1e-7, 1 - 1e-7)
        return np.stack([1 - probs, probs], axis=1)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(np.float32)
    

est = TorchSKClassifier(input_dim=X.shape[1], epochs=20, batch_size=32, lr=1e-3, seed=42)
validator = TrustCVValidator(method="kfold", n_splits=5, random_state=42, shuffle=True)
val_result = validator.validate(model=est, X=X, y=y)
print(val_result.summary())


=== Trustworthy Cross-Validation Results ===

Performance Metrics (mean +/- std) (method: bootstrap):
  accuracy: 0.900 +/- 0.031 [95% CI (bootstrap): 0.875-0.926]
  roc_auc: 0.957 +/- 0.029 [95% CI (bootstrap): 0.936-0.979]
  sensitivity: 0.941 +/- 0.085 [95% CI (bootstrap): 0.864-0.991]
  specificity: 0.841 +/- 0.096 [95% CI (bootstrap): 0.779-0.924]
  precision: 0.909 +/- 0.056 [95% CI (bootstrap): 0.873-0.956]
  recall: 0.941 +/- 0.085 [95% CI (bootstrap): 0.867-0.988]
  f1: 0.921 +/- 0.027 [95% CI (bootstrap): 0.899-0.943]

Data Integrity Checks:
  Leakage Check: PASSED
  Class Balance: PASSED



### Diffrence in Performance Metrics

**Determinism risk:** First is fully deterministic per run due to generator=self._gen; second might see minor variability from sampler order because it doesn’t pass an explicit generator.